<a href="https://colab.research.google.com/github/Le2se0hy/XAI_An/blob/main/OLSSLR_%EC%B5%9C%EC%A2%85(%2Bvif).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 서울 아파트 연도별 OLS / Spatial Lag Regression
# 분석 연도: 2022년, 2023년 각각 별도 분석
#
# 공간가중치:
# - 개별 거래가 분석 단위
# - 동일 좌표 거래는 서로의 이웃에서 제외
# - 1km 이내 다른 좌표 거래를 공간이웃으로 정의
# - 거리역수 가중치 사용
# - 거래 단위 행 표준화
# - 이웃이 없는 거래는 Wy와 WX를 0으로 처리하고 분석은 계속 진행
#
# SLR:
# - 공간시차종속변수 Wy를 포함
# - WX를 도구변수로 사용하는 Spatial 2SLS
#
# RMSE:
# - OLS: 실제 y와 OLS 적합값 비교
# - SLR: 실제 y와 reduced-form 예측값 비교
# ============================================================


# ============================================================
# 0. 라이브러리 설치
# ============================================================
!pip -q install openpyxl statsmodels scikit-learn scipy spreg


# ============================================================
# 1. 라이브러리 불러오기
# ============================================================
import os
import gc
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm

from collections import OrderedDict
from google.colab import drive
from sklearn.neighbors import BallTree

from scipy import sparse
from scipy.linalg import qr
from scipy.sparse.linalg import spsolve, MatrixRankWarning

from spreg import TSLS

warnings.filterwarnings("ignore", category=FutureWarning)


# ============================================================
# 2. 사용자 설정
# ============================================================

FILE_PATH = "/content/drive/MyDrive/Seoul_Aprtment_FINAL.xlsx"
OUTPUT_DIR = "/content/drive/MyDrive"

# 2022년과 2023년을 각각 별도로 분석
TARGET_YEARS = [2022, 2023]

TARGET_COL = "Log_Price_per_m2"
LAT_COL = "Latitude"
LON_COL = "Longitude"

# 공간이웃 거리 기준
DIST_BAND_KM = 1.0

# OLS: HC0 강건 표준오차
# SLR: White 강건 표준오차
USE_ROBUST_SE = True

FEATURE_COLS = [
    "Population",
    "Sex_ratio",
    "Pop. Density",
    "Median age Population",
    "Young Population",
    "Parking_per_Household",
    "Log_Dist_Water",
    "Log_Dist_Green",
    "Log_Dist_Subway",
    "Dist_CBD",
    "max_floor",
    "heating_dummy",
    "num_of_people",
    "Bus_Stop",
    "High_School_Count",
    "Floor",
    "Size_m2",
    "Construction_Year",
    "Spring",
    "Fall",
    "Winter"
]

# 비율 변수
# 값의 최대 절댓값이 2보다 크면 퍼센트 단위로 보고 100으로 나눔
RATIO_COLS = [
    "Sex_ratio",
    "Median age Population",
    "Young Population",
    "Old Population"
]


# ============================================================
# 3. Google Drive 연결 및 데이터 불러오기
# ============================================================
drive.mount("/content/drive", force_remount=True)

df_raw = pd.read_excel(FILE_PATH)

print("=" * 80)
print("원본 데이터 shape:", df_raw.shape)
print("원본 컬럼 수:", len(df_raw.columns))
print("=" * 80)


# ============================================================
# 4. 숫자형 정리 함수
# ============================================================
def clean_numeric_series(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace("\u00a0", "", regex=False)
        .replace({
            "-": np.nan,
            "": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "null": np.nan,
            "NULL": np.nan
        }),
        errors="coerce"
    )


# ============================================================
# 5. 데이터 전처리 함수
# ============================================================
def prepare_data(df_input):
    df = df_input.copy()

    numeric_candidates = [
        "Year_Sold",
        "Month_Sold",
        "Log_Price_per_m2",
        "Population",
        "Sex_ratio",
        "Pop. Density",
        "Median age Population",
        "Young Population",
        "Old Population",
        "Parking_per_Household",
        "Log_Dist_Water",
        "Log_Dist_Green",
        "Log_Dist_Subway",
        "Dist_CBD",
        "max_floor",
        "num_of_people",
        "Bus_Stop",
        "High_School_Count",
        "Latitude",
        "Longitude",
        "Floor",
        "Size_m2",
        "Construction_Year"
    ]

    for col in numeric_candidates:
        if col in df.columns:
            df[col] = clean_numeric_series(df[col])

    # --------------------------------------------------------
    # 난방 더미
    # 도시가스 또는 숫자 1이면 1
    # 나머지는 0
    # --------------------------------------------------------
    if "heating" not in df.columns:
        raise ValueError("heating 컬럼이 없습니다.")

    raw_heating = df_input["heating"].astype(str).str.strip()

    df["heating_dummy"] = raw_heating.apply(
        lambda x: 1
        if (
            "도시가스" in x
            or x == "1"
            or x == "1.0"
        )
        else 0
    )

    # --------------------------------------------------------
    # 분석 연도와 분석 월
    # --------------------------------------------------------
    if "Year_Sold" not in df.columns:
        raise ValueError("Year_Sold 컬럼이 없습니다.")

    if "Month_Sold" not in df.columns:
        raise ValueError("Month_Sold 컬럼이 없습니다.")

    df["Analysis_Year"] = clean_numeric_series(df["Year_Sold"])
    df["Analysis_Month"] = clean_numeric_series(df["Month_Sold"])

    # --------------------------------------------------------
    # 계절 더미
    # 기준 범주: Summer
    # --------------------------------------------------------
    df["Spring"] = (
        df["Analysis_Month"].isin([3, 4, 5]).astype(int)
    )

    df["Fall"] = (
        df["Analysis_Month"].isin([9, 10, 11]).astype(int)
    )

    df["Winter"] = (
        df["Analysis_Month"].isin([12, 1, 2]).astype(int)
    )

    # --------------------------------------------------------
    # 비율 변수 단위 통일
    # 예: 15.2 → 0.152
    # 이미 0~1 범위이면 그대로 유지
    # --------------------------------------------------------
    print("\n[비율 변수 변환 확인]")

    for col in RATIO_COLS:
        if col not in df.columns:
            continue

        valid_values = df[col].dropna()

        if len(valid_values) == 0:
            continue

        original_max = valid_values.abs().max()

        if original_max > 2:
            df[col] = df[col] / 100.0
            print(f"{col}: 100으로 나누어 비율 단위로 변환")
        else:
            print(f"{col}: 기존 단위 유지")

    return df


# ============================================================
# 6. 유의성 표시 함수
# ============================================================
def significance_marker(p_value):
    if pd.isna(p_value):
        return ""
    elif p_value < 0.01:
        return "‡"
    elif p_value < 0.05:
        return "†"
    elif p_value < 0.10:
        return "*"
    else:
        return ""


# ============================================================
# 7. 좌표 그룹 기반 공간가중 연산자 생성
# ============================================================
def build_spatial_operator(
    df_sub,
    lat_col="Latitude",
    lon_col="Longitude",
    dist_band_km=1.0
):
    """
    공간가중치 구성 방식

    1. 분석 단위는 개별 거래
    2. 동일 좌표 거래는 서로의 공간이웃에서 제외
    3. 1km 이내 다른 좌표 거래만 이웃으로 포함
    4. 거리역수 가중치 사용
    5. 거래 단위 행 표준화
    6. 이웃이 없는 좌표는 공간시차를 0으로 처리

    계산 효율을 위해 동일 좌표 거래를 좌표 그룹으로 묶어서 처리함.
    """

    earth_radius_km = 6371.0

    all_coords = df_sub[
        [lat_col, lon_col]
    ].to_numpy(dtype=float)

    # 동일 좌표를 하나의 좌표 그룹으로 묶음
    unique_coords, coord_id = np.unique(
        all_coords,
        axis=0,
        return_inverse=True
    )

    n_obs = len(df_sub)
    n_groups = len(unique_coords)

    if n_groups < 2:
        raise ValueError(
            "유일 좌표가 2개 미만이므로 공간가중치를 만들 수 없습니다."
        )

    # 좌표 그룹별 거래 수
    group_counts = np.bincount(
        coord_id,
        minlength=n_groups
    ).astype(float)

    coords_radian = np.deg2rad(unique_coords)

    tree = BallTree(
        coords_radian,
        metric="haversine"
    )

    radius_radian = (
        dist_band_km / earth_radius_km
    ) * 1.000001

    neighbor_indices, neighbor_distances = tree.query_radius(
        coords_radian,
        r=radius_radian,
        return_distance=True,
        sort_results=False
    )

    row_parts = []
    col_parts = []
    weight_parts = []

    neighboring_coordinate_counts = np.zeros(
        n_groups,
        dtype=int
    )

    neighboring_transaction_counts = np.zeros(
        n_groups,
        dtype=int
    )

    island_groups = []

    for group_index, (idx, dist_rad) in enumerate(
        zip(neighbor_indices, neighbor_distances)
    ):
        distance_km = dist_rad * earth_radius_km

        # 자기 좌표 그룹 제외
        valid_mask = (
            (idx != group_index)
            & (distance_km > 1e-12)
            & (distance_km <= dist_band_km)
        )

        valid_neighbors = idx[valid_mask].astype(int)
        valid_distances = distance_km[valid_mask]

        # ----------------------------------------------------
        # 이웃이 없는 경우
        # 오류를 발생시키지 않고 공간시차를 0으로 처리
        # ----------------------------------------------------
        if len(valid_neighbors) == 0:
            island_groups.append(group_index)
            continue

        inverse_distance = 1.0 / valid_distances

        # 각 좌표에 속한 개별 거래 수까지 고려한
        # 거래 단위 행 표준화 분모
        denominator = np.sum(
            inverse_distance
            * group_counts[valid_neighbors]
        )

        if (
            denominator <= 0
            or not np.isfinite(denominator)
        ):
            island_groups.append(group_index)
            continue

        normalized_weights = (
            inverse_distance / denominator
        )

        row_parts.append(
            np.full(
                len(valid_neighbors),
                group_index,
                dtype=int
            )
        )

        col_parts.append(valid_neighbors)
        weight_parts.append(normalized_weights)

        neighboring_coordinate_counts[group_index] = (
            len(valid_neighbors)
        )

        neighboring_transaction_counts[group_index] = int(
            np.sum(group_counts[valid_neighbors])
        )

    # 모든 좌표가 이웃이 없는 경우에는 분석 불가
    if len(island_groups) == n_groups:
        raise ValueError(
            f"모든 좌표가 {dist_band_km}km 이내 공간이웃이 없습니다. "
            "DIST_BAND_KM 값을 확대해야 합니다."
        )

    # 이웃이 있는 좌표의 공간가중치만 희소행렬에 저장
    rows = np.concatenate(row_parts)
    cols = np.concatenate(col_parts)
    weights = np.concatenate(weight_parts)

    group_weight_matrix = sparse.csr_matrix(
        (weights, (rows, cols)),
        shape=(n_groups, n_groups)
    )

    # --------------------------------------------------------
    # 행 표준화 검증
    # 이웃이 있는 좌표는 행 합 1
    # 이웃이 없는 좌표는 행 합 0
    # --------------------------------------------------------
    transaction_row_sums = np.asarray(
        group_weight_matrix @ group_counts
    ).reshape(-1)

    non_island_mask = np.ones(
        n_groups,
        dtype=bool
    )

    if len(island_groups) > 0:
        non_island_mask[
            np.asarray(island_groups, dtype=int)
        ] = False

    row_sum_max_error = np.max(
        np.abs(
            transaction_row_sums[non_island_mask] - 1.0
        )
    )

    if row_sum_max_error > 1e-8:
        raise ValueError(
            "이웃이 있는 좌표의 공간가중치 행 표준화에 실패했습니다. "
            f"최대 오차: {row_sum_max_error}"
        )

    if len(island_groups) > 0:
        island_indices = np.asarray(
            island_groups,
            dtype=int
        )

        island_row_sums = transaction_row_sums[
            island_indices
        ]

        if not np.allclose(
            island_row_sums,
            0.0,
            atol=1e-12
        ):
            raise ValueError(
                "이웃이 없는 좌표의 공간가중치가 0이 아닙니다."
            )

        island_transaction_count = int(
            np.sum(group_counts[island_indices])
        )

        print(
            f"[경고] {dist_band_km}km 이내 다른 좌표의 이웃이 없는 "
            f"좌표 그룹이 {len(island_groups)}개이며, "
            f"해당 거래는 {island_transaction_count}건입니다."
        )

        print(
            "[처리] 해당 거래는 제거하지 않고 "
            "Wy와 WX를 0으로 처리합니다."
        )

    else:
        island_transaction_count = 0

    spatial_info = {
        "n_obs": n_obs,
        "n_groups": n_groups,
        "coord_id": coord_id,
        "group_counts": group_counts,
        "group_weight_matrix": group_weight_matrix,
        "neighbor_coordinate_counts": (
            neighboring_coordinate_counts
        ),
        "neighbor_transaction_counts": (
            neighboring_transaction_counts
        ),
        "row_sum_max_error": row_sum_max_error,
        "n_group_links": int(group_weight_matrix.nnz),
        "island_groups": island_groups,
        "n_island_groups": len(island_groups),
        "n_island_transactions": island_transaction_count
    }

    return spatial_info


# ============================================================
# 8. 공간시차 계산 함수
# ============================================================
def calculate_spatial_lag(values, spatial_info):
    """
    1차원 벡터 또는 2차원 행렬에 대해 공간시차 Wv를 계산함.

    이웃이 없는 좌표는 공간가중행렬의 해당 행이 모두 0이므로
    자동으로 공간시차가 0이 됨.
    """

    value_array = np.asarray(
        values,
        dtype=float
    )

    was_one_dimensional = (
        value_array.ndim == 1
    )

    if was_one_dimensional:
        value_array = value_array.reshape(-1, 1)

    coord_id = spatial_info["coord_id"]
    group_counts = spatial_info["group_counts"]
    group_weight_matrix = spatial_info[
        "group_weight_matrix"
    ]

    n_groups = len(group_counts)
    n_columns = value_array.shape[1]

    # 좌표 그룹별 거래값 합계
    group_sums = np.zeros(
        (n_groups, n_columns),
        dtype=float
    )

    np.add.at(
        group_sums,
        coord_id,
        value_array
    )

    # 좌표 그룹별 공간시차
    group_lag_values = (
        group_weight_matrix @ group_sums
    )

    # 개별 거래 행으로 다시 확장
    transaction_lag_values = np.asarray(
        group_lag_values
    )[coord_id]

    if was_one_dimensional:
        return transaction_lag_values.reshape(-1)

    return transaction_lag_values


# ============================================================
# 9. 사용 가능한 공간 도구변수 WX 선택
# ============================================================
def select_valid_instruments(
    x_array,
    q_array,
    q_names,
    tolerance=1e-10
):
    """
    WX 중 다음 변수를 제거함.

    1. 상수에 가까운 변수
    2. 기존 X로 완전히 설명되는 변수
    3. 다른 WX 변수와 완전한 선형종속 관계인 변수
    """

    x_array = np.asarray(
        x_array,
        dtype=float
    )

    q_array = np.asarray(
        q_array,
        dtype=float
    )

    x_with_constant = np.column_stack([
        np.ones(len(x_array)),
        x_array
    ])

    # WX에서 기존 X로 설명되는 부분 제거
    projection_coef = np.linalg.lstsq(
        x_with_constant,
        q_array,
        rcond=None
    )[0]

    q_residual = (
        q_array
        - x_with_constant @ projection_coef
    )

    residual_norm = np.linalg.norm(
        q_residual,
        axis=0
    )

    nonconstant_mask = (
        residual_norm > tolerance
    )

    q_array = q_array[:, nonconstant_mask]
    q_residual = q_residual[:, nonconstant_mask]

    filtered_names = [
        name
        for name, keep in zip(
            q_names,
            nonconstant_mask
        )
        if keep
    ]

    if q_array.shape[1] == 0:
        raise ValueError(
            "사용 가능한 공간 도구변수 WX가 없습니다."
        )

    # QR pivoting을 이용한 선형독립 도구변수 선택
    _, r_matrix, pivot_order = qr(
        q_residual,
        mode="economic",
        pivoting=True
    )

    diagonal_values = np.abs(
        np.diag(r_matrix)
    )

    if len(diagonal_values) == 0:
        raise ValueError(
            "공간 도구변수의 행렬 순위를 계산할 수 없습니다."
        )

    rank_tolerance = (
        diagonal_values.max() * 1e-10
    )

    instrument_rank = int(
        np.sum(
            diagonal_values > rank_tolerance
        )
    )

    if instrument_rank == 0:
        raise ValueError(
            "공간 도구변수 WX가 모두 선형종속입니다."
        )

    selected_indices = np.sort(
        pivot_order[:instrument_rank]
    )

    selected_q = q_array[
        :,
        selected_indices
    ]

    selected_names = [
        filtered_names[index]
        for index in selected_indices
    ]

    return selected_q, selected_names


# ============================================================
# 10. Spatial Lag reduced-form 예측값 계산
# ============================================================
def calculate_reduced_form_prediction(
    xb,
    rho,
    spatial_info
):
    """
    Spatial Lag Model

        y = rho Wy + X beta + error

    reduced form

        y_hat = (I - rho W)^(-1) X beta

    전체 거래 단위 W를 직접 만들지 않고,
    좌표 그룹 단위 희소행렬을 이용하여 계산함.

    이웃이 없는 좌표는 W의 해당 행이 0이므로
    그 거래의 예측값은 X beta가 됨.
    """

    if not np.isfinite(rho):
        raise ValueError(
            "추정된 rho가 유한한 값이 아닙니다."
        )

    if abs(rho) >= 1:
        raise ValueError(
            f"추정된 rho={rho:.6f}가 |rho|<1 범위를 벗어났습니다."
        )

    xb = np.asarray(
        xb,
        dtype=float
    ).reshape(-1)

    coord_id = spatial_info["coord_id"]
    group_counts = spatial_info["group_counts"]
    group_weight_matrix = spatial_info[
        "group_weight_matrix"
    ]

    n_groups = len(group_counts)

    # 좌표 그룹별 X beta 합계
    group_xb_sum = np.bincount(
        coord_id,
        weights=xb,
        minlength=n_groups
    )

    count_matrix = sparse.diags(
        group_counts,
        format="csr"
    )

    reduced_form_matrix = (
        sparse.eye(
            n_groups,
            format="csr"
        )
        - rho
        * (
            count_matrix
            @ group_weight_matrix
        )
    )

    with warnings.catch_warnings():
        warnings.simplefilter(
            "error",
            MatrixRankWarning
        )

        try:
            predicted_group_sums = spsolve(
                reduced_form_matrix,
                group_xb_sum
            )

        except MatrixRankWarning as error:
            raise ValueError(
                "Reduced-form 예측 과정에서 행렬이 특이행렬이 되었습니다."
            ) from error

    if not np.all(
        np.isfinite(predicted_group_sums)
    ):
        raise ValueError(
            "Reduced-form 예측값에 무한대 또는 결측값이 발생했습니다."
        )

    predicted_group_lag = np.asarray(
        group_weight_matrix
        @ predicted_group_sums
    ).reshape(-1)

    predicted_y = (
        xb
        + rho
        * predicted_group_lag[coord_id]
    )

    return predicted_y


# ============================================================
# 11. 회귀계수 비교표 생성
# ============================================================
def build_coefficient_table(
    ols_model,
    slr_model,
    feature_cols,
    target_col
):
    # --------------------------------------------------------
    # OLS 결과
    # --------------------------------------------------------
    ols_names = [
        "const"
    ] + feature_cols

    ols_coef = np.asarray(
        ols_model.params
    ).reshape(-1)

    ols_std_error = np.asarray(
        ols_model.bse
    ).reshape(-1)

    ols_stat = np.asarray(
        ols_model.tvalues
    ).reshape(-1)

    ols_pvalue = np.asarray(
        ols_model.pvalues
    ).reshape(-1)

    ols_table = pd.DataFrame({
        "Variable": ols_names,
        "OLS_Coef": ols_coef,
        "OLS_Std_Error": ols_std_error,
        "OLS_Statistic": ols_stat,
        "OLS_pvalue": ols_pvalue,
        "OLS_sig": [
            significance_marker(p)
            for p in ols_pvalue
        ]
    })

    # --------------------------------------------------------
    # SLR 결과
    # 순서: 상수항, 설명변수, 공간시차계수
    # --------------------------------------------------------
    slr_names = (
        ["const"]
        + feature_cols
        + [f"W_{target_col}"]
    )

    slr_coef = np.asarray(
        slr_model.betas
    ).reshape(-1)

    slr_std_error = np.asarray(
        slr_model.std_err
    ).reshape(-1)

    slr_stat = np.array([
        result[0]
        for result in slr_model.z_stat
    ])

    slr_pvalue = np.array([
        result[1]
        for result in slr_model.z_stat
    ])

    if len(slr_names) != len(slr_coef):
        raise ValueError(
            "SLR 계수 개수와 변수명 개수가 일치하지 않습니다."
        )

    slr_table = pd.DataFrame({
        "Variable": slr_names,
        "SLR_Coef": slr_coef,
        "SLR_Std_Error": slr_std_error,
        "SLR_Statistic": slr_stat,
        "SLR_pvalue": slr_pvalue,
        "SLR_sig": [
            significance_marker(p)
            for p in slr_pvalue
        ]
    })

    coefficient_table = pd.merge(
        ols_table,
        slr_table,
        on="Variable",
        how="outer"
    )

    return coefficient_table


# ============================================================
# 12. 논문 형식 비교표 생성
# ============================================================
def build_thesis_table(
    coefficient_table,
    performance_table
):
    variable_mapping = OrderedDict([
        ("--- Property Characteristics ---", None),

        ("Size", "Size_m2"),
        ("Floor", "Floor"),
        ("Units", "num_of_people"),
        ("Parking", "Parking_per_Household"),
        ("Construction Year", "Construction_Year"),
        ("Max Floor", "max_floor"),
        ("Heating Dummy", "heating_dummy"),

        ("--- Accessibility & Environment ---", None),

        ("Dist. Subway (Log)", "Log_Dist_Subway"),
        ("Dist. Green (Log)", "Log_Dist_Green"),
        ("Dist. Water (Log)", "Log_Dist_Water"),
        ("Dist. CBD", "Dist_CBD"),
        ("Bus Stop", "Bus_Stop"),
        ("High School Count", "High_School_Count"),

        ("--- Demographics ---", None),

        ("Population", "Population"),
        ("Young Population", "Young Population"),
        ("Sex Ratio", "Sex_ratio"),
        ("Pop. Density", "Pop. Density"),
        ("Median Age Population", "Median age Population"),

        ("--- Seasonality ---", None),

        ("Spring", "Spring"),
        ("Fall", "Fall"),
        ("Winter", "Winter"),

        ("Constant", "const"),

        (
            "Spatial Lag Coefficient",
            f"W_{TARGET_COL}"
        )
    ])

    thesis_table = pd.DataFrame(
        index=list(variable_mapping.keys()),
        columns=["OLS", "SLR"]
    )

    indexed_coef = coefficient_table.set_index(
        "Variable"
    )

    for row_name, variable_name in variable_mapping.items():

        if variable_name is None:
            thesis_table.loc[
                row_name,
                ["OLS", "SLR"]
            ] = ""

            continue

        if variable_name not in indexed_coef.index:
            thesis_table.loc[
                row_name,
                ["OLS", "SLR"]
            ] = "–"

            continue

        row = indexed_coef.loc[variable_name]

        if pd.notna(
            row.get("OLS_Coef", np.nan)
        ):
            thesis_table.loc[
                row_name,
                "OLS"
            ] = (
                f"{row['OLS_Coef']:.4f}"
                f"{row['OLS_sig']}"
            )

        else:
            thesis_table.loc[
                row_name,
                "OLS"
            ] = "–"

        if pd.notna(
            row.get("SLR_Coef", np.nan)
        ):
            thesis_table.loc[
                row_name,
                "SLR"
            ] = (
                f"{row['SLR_Coef']:.4f}"
                f"{row['SLR_sig']}"
            )

        else:
            thesis_table.loc[
                row_name,
                "SLR"
            ] = "–"

    ols_perf = performance_table[
        performance_table["Model"] == "OLS"
    ].iloc[0]

    slr_perf = performance_table[
        performance_table["Model"] == "SLR"
    ].iloc[0]

    ols_f_sig = significance_marker(
        ols_perf["Prob_F"]
    )

    additional_rows = pd.DataFrame(
        {
            "OLS": [
                f"{ols_perf['F_stat']:.2f}{ols_f_sig}",
                f"{ols_perf['RMSE']:.4f}",
                f"{ols_perf['R2']:.4f}",
                f"{ols_perf['Adjusted_R2']:.4f}",
                "–"
            ],

            "SLR": [
                "–",
                f"{slr_perf['RMSE']:.4f}",
                "–",
                "–",
                f"{slr_perf['Spatial_Pseudo_R2']:.4f}"
            ]
        },

        index=[
            "F-statistics",
            "RMSE",
            "R2",
            "Adjusted R2",
            "Spatial Pseudo R2"
        ]
    )

    thesis_table = pd.concat([
        thesis_table,
        additional_rows
    ])

    return thesis_table


# ============================================================
# 13. 연도별 OLS / SLR 실행 함수
# ============================================================
def run_models_for_year(
    df_input,
    year_value,
    feature_cols,
    target_col,
    lat_col,
    lon_col,
    dist_band_km
):
    print("\n")
    print("=" * 80)
    print(f"{year_value}년 분석 시작")
    print("=" * 80)

    df_year = df_input[
        df_input["Analysis_Year"] == year_value
    ].copy()

    df_year["_Original_Index"] = df_year.index

    required_cols = (
        feature_cols
        + [
            target_col,
            lat_col,
            lon_col
        ]
    )

    missing_columns = [
        col
        for col in required_cols
        if col not in df_year.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{year_value}년 분석에 필요한 컬럼이 없습니다: "
            f"{missing_columns}"
        )

    print(
        f"[{year_value}] 결측 제거 전 거래 수:",
        len(df_year)
    )

    print(
        f"\n[{year_value}] 변수별 결측치 상위 10개"
    )

    print(
        df_year[required_cols]
        .isna()
        .sum()
        .sort_values(ascending=False)
        .head(10)
    )

    # 분석에 필요한 변수의 결측 행 제거
    df_year = (
        df_year
        .dropna(subset=required_cols)
        .reset_index(drop=True)
    )

    print(
        f"\n[{year_value}] 최종 분석 거래 수:",
        len(df_year)
    )

    if len(df_year) < 30:
        raise ValueError(
            f"{year_value}년 분석 데이터가 30건 미만입니다."
        )

    # --------------------------------------------------------
    # 설명변수와 종속변수 구성
    # --------------------------------------------------------
    x_array = df_year[
        feature_cols
    ].to_numpy(dtype=float)

    y_array = df_year[
        target_col
    ].to_numpy(dtype=float).reshape(-1, 1)

    x_with_constant = sm.add_constant(
        df_year[feature_cols],
        has_constant="add"
    )

    # 완전한 다중공선성 확인
    expected_rank = x_with_constant.shape[1]

    actual_rank = np.linalg.matrix_rank(
        x_with_constant.to_numpy(dtype=float)
    )

    if actual_rank < expected_rank:
        raise ValueError(
            f"설명변수 행렬에 완전한 다중공선성이 있습니다. "
            f"행렬 rank={actual_rank}, 변수 수={expected_rank}"
        )

    # --------------------------------------------------------
    # OLS 추정
    # --------------------------------------------------------
    if USE_ROBUST_SE:
        ols_model = sm.OLS(
            y_array.reshape(-1),
            x_with_constant
        ).fit(
            cov_type="HC0"
        )

    else:
        ols_model = sm.OLS(
            y_array.reshape(-1),
            x_with_constant
        ).fit()

    ols_prediction = np.asarray(
        ols_model.predict(
            x_with_constant
        )
    ).reshape(-1)

    ols_residual = (
        y_array.reshape(-1)
        - ols_prediction
    )

    ols_rmse = np.sqrt(
        np.mean(
            ols_residual ** 2
        )
    )

    # --------------------------------------------------------
    # 공간가중치 생성
    # --------------------------------------------------------
    print(
        f"\n[{year_value}] "
        f"{dist_band_km}km 공간가중치 생성 중"
    )

    spatial_info = build_spatial_operator(
        df_year,
        lat_col=lat_col,
        lon_col=lon_col,
        dist_band_km=dist_band_km
    )

    non_island_group_mask = (
        spatial_info["neighbor_coordinate_counts"] > 0
    )

    if np.any(non_island_group_mask):
        mean_neighbor_coordinates = (
            spatial_info[
                "neighbor_coordinate_counts"
            ][non_island_group_mask].mean()
        )

        median_neighbor_coordinates = np.median(
            spatial_info[
                "neighbor_coordinate_counts"
            ][non_island_group_mask]
        )

        mean_neighbor_transactions = (
            spatial_info[
                "neighbor_transaction_counts"
            ][non_island_group_mask].mean()
        )

        median_neighbor_transactions = np.median(
            spatial_info[
                "neighbor_transaction_counts"
            ][non_island_group_mask]
        )

    else:
        mean_neighbor_coordinates = 0.0
        median_neighbor_coordinates = 0.0
        mean_neighbor_transactions = 0.0
        median_neighbor_transactions = 0.0

    print(
        f"[{year_value}] 유일 좌표 수:",
        spatial_info["n_groups"]
    )

    print(
        f"[{year_value}] 좌표 그룹 연결 수:",
        spatial_info["n_group_links"]
    )

    print(
        f"[{year_value}] 이웃 없는 좌표 그룹 수:",
        spatial_info["n_island_groups"]
    )

    print(
        f"[{year_value}] 이웃 없는 거래 수:",
        spatial_info["n_island_transactions"]
    )

    print(
        f"[{year_value}] 이웃이 있는 좌표의 평균 이웃 좌표 수:",
        mean_neighbor_coordinates
    )

    print(
        f"[{year_value}] 이웃이 있는 좌표의 평균 이웃 거래 수:",
        mean_neighbor_transactions
    )

    print(
        f"[{year_value}] 행 표준화 최대 오차:",
        spatial_info["row_sum_max_error"]
    )

    # --------------------------------------------------------
    # Wy 및 WX 계산
    # --------------------------------------------------------
    wy_array = calculate_spatial_lag(
        y_array.reshape(-1),
        spatial_info
    ).reshape(-1, 1)

    wx_array = calculate_spatial_lag(
        x_array,
        spatial_info
    )

    wx_names = [
        f"W_{col}"
        for col in feature_cols
    ]

    # 사용할 수 있는 공간 도구변수 선택
    q_array, q_names = select_valid_instruments(
        x_array=x_array,
        q_array=wx_array,
        q_names=wx_names
    )

    print(
        f"[{year_value}] 사용된 공간 도구변수 수:",
        q_array.shape[1]
    )

    print(
        f"[{year_value}] 사용된 공간 도구변수:",
        q_names
    )

    # --------------------------------------------------------
    # Spatial Lag Regression
    # Wy를 WX로 도구변수화한 Spatial 2SLS
    # --------------------------------------------------------
    slr_model = TSLS(
        y=y_array,
        x=x_array,
        yend=wy_array,
        q=q_array,

        robust=(
            "white"
            if USE_ROBUST_SE
            else None
        ),

        name_y=target_col,
        name_x=feature_cols,
        name_yend=[f"W_{target_col}"],
        name_q=q_names,
        name_ds=f"Seoul Apartment {year_value}"
    )

    slr_betas = np.asarray(
        slr_model.betas
    ).reshape(-1)

    # 계수 순서
    # 상수항, 설명변수 계수들, rho
    slr_intercept = slr_betas[0]

    slr_feature_betas = slr_betas[
        1:1 + len(feature_cols)
    ]

    estimated_rho = slr_betas[-1]

    xb = (
        slr_intercept
        + x_array @ slr_feature_betas
    )

    # --------------------------------------------------------
    # SLR reduced-form 예측값 및 RMSE
    # --------------------------------------------------------
    slr_prediction = calculate_reduced_form_prediction(
        xb=xb,
        rho=estimated_rho,
        spatial_info=spatial_info
    )

    slr_residual = (
        y_array.reshape(-1)
        - slr_prediction
    )

    slr_rmse = np.sqrt(
        np.mean(
            slr_residual ** 2
        )
    )

    # 공간 Pseudo R2
    if np.std(slr_prediction) > 0:
        slr_spatial_pseudo_r2 = (
            np.corrcoef(
                y_array.reshape(-1),
                slr_prediction
            )[0, 1] ** 2
        )

    else:
        slr_spatial_pseudo_r2 = np.nan

    rho_pvalue = float(
        slr_model.z_stat[-1][1]
    )

    # --------------------------------------------------------
    # 성능표
    # --------------------------------------------------------
    performance_table = pd.DataFrame([
        {
            "Year": year_value,
            "Model": "OLS",
            "N": len(df_year),

            "RMSE": ols_rmse,

            "R2": ols_model.rsquared,
            "Adjusted_R2": ols_model.rsquared_adj,

            "Spatial_Pseudo_R2": np.nan,

            "AIC": ols_model.aic,
            "BIC": ols_model.bic,

            "F_stat": ols_model.fvalue,
            "Prob_F": ols_model.f_pvalue,

            "Rho": np.nan,
            "Rho_pvalue": np.nan,

            "Robust_SE": (
                "HC0"
                if USE_ROBUST_SE
                else "No"
            ),

            "Island_Groups": (
                spatial_info["n_island_groups"]
            ),

            "Island_Transactions": (
                spatial_info["n_island_transactions"]
            )
        },

        {
            "Year": year_value,
            "Model": "SLR",
            "N": len(df_year),

            "RMSE": slr_rmse,

            "R2": np.nan,
            "Adjusted_R2": np.nan,

            "Spatial_Pseudo_R2": (
                slr_spatial_pseudo_r2
            ),

            "AIC": np.nan,
            "BIC": np.nan,

            "F_stat": np.nan,
            "Prob_F": np.nan,

            "Rho": estimated_rho,
            "Rho_pvalue": rho_pvalue,

            "Robust_SE": (
                "White"
                if USE_ROBUST_SE
                else "No"
            ),

            "Island_Groups": (
                spatial_info["n_island_groups"]
            ),

            "Island_Transactions": (
                spatial_info["n_island_transactions"]
            )
        }
    ])

    # --------------------------------------------------------
    # 회귀계수표
    # --------------------------------------------------------
    coefficient_table = build_coefficient_table(
        ols_model=ols_model,
        slr_model=slr_model,
        feature_cols=feature_cols,
        target_col=target_col
    )

    # --------------------------------------------------------
    # 논문용 비교표
    # --------------------------------------------------------
    thesis_table = build_thesis_table(
        coefficient_table=coefficient_table,
        performance_table=performance_table
    )

    # --------------------------------------------------------
    # 예측값표
    # --------------------------------------------------------
    coord_id = spatial_info["coord_id"]

    island_group_set = set(
        spatial_info["island_groups"]
    )

    is_island_transaction = np.array([
        int(group_id in island_group_set)
        for group_id in coord_id
    ])

    prediction_table = pd.DataFrame({
        "Original_Index": df_year[
            "_Original_Index"
        ].values,

        "Latitude": df_year[
            lat_col
        ].values,

        "Longitude": df_year[
            lon_col
        ].values,

        "Is_Island_Transaction": (
            is_island_transaction
        ),

        "Actual": y_array.reshape(-1),

        "OLS_Pred": ols_prediction,
        "OLS_Residual": ols_residual,

        "SLR_Pred_ReducedForm": slr_prediction,
        "SLR_Residual": slr_residual,

        "WY_Observed": wy_array.reshape(-1)
    })

    # --------------------------------------------------------
    # 공간가중치 정보표
    # --------------------------------------------------------
    spatial_summary_table = pd.DataFrame([
        {
            "Year": year_value,

            "Distance_Band_km": dist_band_km,

            "Number_of_Transactions": len(df_year),

            "Number_of_Unique_Coordinates": (
                spatial_info["n_groups"]
            ),

            "Coordinate_Group_Links": (
                spatial_info["n_group_links"]
            ),

            "Island_Coordinate_Groups": (
                spatial_info["n_island_groups"]
            ),

            "Island_Transactions": (
                spatial_info["n_island_transactions"]
            ),

            "Island_Handling": (
                "Retained; Wy and WX set to zero"
            ),

            "Mean_Neighbor_Coordinates_NonIsland": (
                mean_neighbor_coordinates
            ),

            "Median_Neighbor_Coordinates_NonIsland": (
                median_neighbor_coordinates
            ),

            "Mean_Neighbor_Transactions_NonIsland": (
                mean_neighbor_transactions
            ),

            "Median_Neighbor_Transactions_NonIsland": (
                median_neighbor_transactions
            ),

            "Row_Standardization_Max_Error": (
                spatial_info["row_sum_max_error"]
            ),

            "Same_Coordinate_Transactions_Included": "No",

            "Weight_Type": "Inverse distance",

            "Instrument_Count": q_array.shape[1]
        }
    ])

    # --------------------------------------------------------
    # 유의성 기호 설명표
    # --------------------------------------------------------
    significance_legend = pd.DataFrame({
        "Symbol": ["‡", "†", "*"],

        "p-value": [
            "p < 0.01",
            "p < 0.05",
            "p < 0.10"
        ]
    })

    # --------------------------------------------------------
    # 분석 방법 설명표
    # --------------------------------------------------------
    method_note = pd.DataFrame({
        "Item": [
            "Analysis unit",
            "OLS estimation",
            "SLR estimation",
            "Spatial lag",
            "Spatial instruments",
            "Distance threshold",
            "Weight",
            "Same-coordinate transactions",
            "Island transactions",
            "Row standardization",
            "OLS RMSE",
            "SLR RMSE",
            "SLR Pseudo R2"
        ],

        "Description": [
            "Individual apartment transaction",

            (
                "Ordinary least squares with HC0 robust standard errors"
                if USE_ROBUST_SE
                else "Ordinary least squares"
            ),

            (
                "Spatial two-stage least squares with White robust standard errors"
                if USE_ROBUST_SE
                else "Spatial two-stage least squares"
            ),

            "Spatially lagged dependent variable Wy",

            "Spatially lagged explanatory variables WX",

            f"{dist_band_km} km",

            "Inverse-distance weight",

            (
                "Transactions at the same coordinates were excluded "
                "from each other's spatial neighbors"
            ),

            (
                "Transactions with no neighbor within the distance threshold "
                "were retained, and their Wy and WX were set to zero"
            ),

            "Transaction-level row standardization",

            (
                "RMSE between observed y and OLS fitted values"
            ),

            (
                "RMSE between observed y and reduced-form SLR prediction: "
                "(I-rho W)^(-1)X beta"
            ),

            (
                "Squared correlation between observed y and "
                "reduced-form SLR prediction"
            )
        ]
    })

    # --------------------------------------------------------
    # 결과 출력
    # --------------------------------------------------------
    print(
        f"\n[{year_value}] 분석 결과"
    )

    print(
        f"[{year_value}] OLS RMSE:",
        round(ols_rmse, 6)
    )

    print(
        f"[{year_value}] SLR RMSE:",
        round(slr_rmse, 6)
    )

    print(
        f"[{year_value}] 추정 rho:",
        round(estimated_rho, 6)
    )

    print(
        f"[{year_value}] rho p-value:",
        rho_pvalue
    )

    display(performance_table)
    display(coefficient_table)
    display(thesis_table)

    return {
        "year": year_value,
        "data": df_year,

        "ols_model": ols_model,
        "slr_model": slr_model,

        "performance_table": performance_table,
        "coefficient_table": coefficient_table,
        "thesis_table": thesis_table,
        "prediction_table": prediction_table,

        "spatial_summary_table": (
            spatial_summary_table
        ),

        "significance_legend": (
            significance_legend
        ),

        "method_note": method_note,
        "q_names": q_names
    }


# ============================================================
# 14. 전체 데이터 전처리
# ============================================================
df_all = prepare_data(df_raw)

missing_features = [
    col
    for col in FEATURE_COLS
    if col not in df_all.columns
]

if missing_features:
    raise ValueError(
        f"설명변수 컬럼이 누락되었습니다: {missing_features}"
    )

available_years = sorted(
    df_all["Analysis_Year"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

print("\n분석 가능한 연도:", available_years)

missing_years = [
    year
    for year in TARGET_YEARS
    if year not in available_years
]

if missing_years:
    raise ValueError(
        f"데이터에 존재하지 않는 분석 연도입니다: {missing_years}"
    )


# ============================================================
# 15. 2022년과 2023년 각각 실행 및 저장
# ============================================================
all_performance_tables = []

for analysis_year in TARGET_YEARS:

    result = run_models_for_year(
        df_input=df_all,
        year_value=analysis_year,

        feature_cols=FEATURE_COLS,

        target_col=TARGET_COL,
        lat_col=LAT_COL,
        lon_col=LON_COL,

        dist_band_km=DIST_BAND_KM
    )

    output_file = os.path.join(
        OUTPUT_DIR,
        f"Seoul_Apartment_OLS_SLR_{analysis_year}.xlsx"
    )

    with pd.ExcelWriter(
        output_file,
        engine="openpyxl"
    ) as writer:

        result[
            "performance_table"
        ].to_excel(
            writer,
            sheet_name="Performance",
            index=False
        )

        result[
            "coefficient_table"
        ].to_excel(
            writer,
            sheet_name="Coefficients",
            index=False
        )

        result[
            "thesis_table"
        ].to_excel(
            writer,
            sheet_name="Thesis_Table",
            index=True
        )

        result[
            "prediction_table"
        ].to_excel(
            writer,
            sheet_name="Predictions",
            index=False
        )

        result[
            "spatial_summary_table"
        ].to_excel(
            writer,
            sheet_name="Spatial_Info",
            index=False
        )

        result[
            "method_note"
        ].to_excel(
            writer,
            sheet_name="Method_Note",
            index=False
        )

        result[
            "significance_legend"
        ].to_excel(
            writer,
            sheet_name="Significance",
            index=False
        )

        pd.DataFrame({
            "Selected_Instrument": result["q_names"]
        }).to_excel(
            writer,
            sheet_name="Instruments",
            index=False
        )

        pd.DataFrame({
            "OLS_Summary": str(
                result["ols_model"].summary()
            ).splitlines()
        }).to_excel(
            writer,
            sheet_name="OLS_Summary",
            index=False
        )

        pd.DataFrame({
            "SLR_Summary": str(
                result["slr_model"].summary
            ).splitlines()
        }).to_excel(
            writer,
            sheet_name="SLR_Summary",
            index=False
        )

    all_performance_tables.append(
        result["performance_table"].copy()
    )

    print(
        f"\n{analysis_year}년 결과 저장 완료:"
    )

    print(output_file)

    del result
    gc.collect()


# ============================================================
# 16. 2022년과 2023년 성능표 통합 저장
# ============================================================
combined_performance = pd.concat(
    all_performance_tables,
    ignore_index=True
)

combined_output_file = os.path.join(
    OUTPUT_DIR,
    "Seoul_Apartment_OLS_SLR_2022_2023_Summary.xlsx"
)

with pd.ExcelWriter(
    combined_output_file,
    engine="openpyxl"
) as writer:

    combined_performance.to_excel(
        writer,
        sheet_name="Combined_Performance",
        index=False
    )


# ============================================================
# 17. 최종 저장 경로 출력
# ============================================================
print("\n")
print("=" * 80)
print("전체 분석 완료")
print("=" * 80)

print("\n2022년 결과 파일:")
print(
    os.path.join(
        OUTPUT_DIR,
        "Seoul_Apartment_OLS_SLR_2022.xlsx"
    )
)

print("\n2023년 결과 파일:")
print(
    os.path.join(
        OUTPUT_DIR,
        "Seoul_Apartment_OLS_SLR_2023.xlsx"
    )
)

print("\n통합 성능 요약 파일:")
print(combined_output_file)

print("=" * 80)

Mounted at /content/drive
원본 데이터 shape: (162375, 41)
원본 컬럼 수: 41

[비율 변수 변환 확인]
Sex_ratio: 기존 단위 유지
Median age Population: 기존 단위 유지
Young Population: 기존 단위 유지
Old Population: 기존 단위 유지

분석 가능한 연도: [2022, 2023, 2024, 2025]


2022년 분석 시작
[2022] 결측 제거 전 거래 수: 10579

[2022] 변수별 결측치 상위 10개
Population               0
Sex_ratio                0
Pop. Density             0
Median age Population    0
Young Population         0
Parking_per_Household    0
Log_Dist_Water           0
Log_Dist_Green           0
Log_Dist_Subway          0
Dist_CBD                 0
dtype: int64

[2022] 최종 분석 거래 수: 10579

[2022] 1.0km 공간가중치 생성 중
[경고] 1.0km 이내 다른 좌표의 이웃이 없는 좌표 그룹이 1개이며, 해당 거래는 1건입니다.
[처리] 해당 거래는 제거하지 않고 Wy와 WX를 0으로 처리합니다.
[2022] 유일 좌표 수: 3167
[2022] 좌표 그룹 연결 수: 91498
[2022] 이웃 없는 좌표 그룹 수: 1
[2022] 이웃 없는 거래 수: 1
[2022] 이웃이 있는 좌표의 평균 이웃 좌표 수: 28.900189513581807
[2022] 이웃이 있는 좌표의 평균 이웃 거래 수: 88.93809222994315
[2022] 행 표준화 최대 오차: 6.661338147750939e-16
[2022] 사용된 공간 도구변수 수: 21
[2022] 사용된 공간 도구변수: ['W_Populati

,Year,Model,N,RMSE,R2,Adjusted_R2,Spatial_Pseudo_R2,AIC,BIC,F_stat,Prob_F,Rho,Rho_pvalue,Robust_SE,Island_Groups,Island_Transactions
0,2022,OLS,10579,0.335102,0.45093,0.449838,NaN,6933.408658,7093.274434,410.831766,0.0,NaN,NaN,HC0,1,1
1,2022,SLR,10579,0.335456,NaN,NaN,0.449945,NaN,NaN,NaN,NaN,0.401726,0.054497,White,1,1


,Variable,OLS_Coef,OLS_Std_Error,OLS_Statistic,OLS_pvalue,OLS_sig,SLR_Coef,SLR_Std_Error,SLR_Statistic,SLR_pvalue,SLR_sig
0,Bus_Stop,-0.005951,6.079695e-04,-9.788969,1.255697e-22,‡,-4.142810e-03,1.104216e-03,-3.751811,1.755616e-04,‡
1,Construction_Year,-0.002020,4.109174e-04,-4.916240,8.822226e-07,‡,-6.254554e-04,6.931733e-04,-0.902307,3.668936e-01,
2,Dist_CBD,0.000008,2.484625e-06,3.408145,6.540611e-04,‡,3.757331e-06,3.321152e-06,1.131334,2.579147e-01,
3,Fall,-0.098319,1.089916e-02,-9.020750,1.868054e-19,‡,-9.588466e-02,9.399861e-03,-10.200646,1.969587e-24,‡
4,Floor,0.001828,6.141817e-04,2.976097,2.919428e-03,‡,2.115574e-03,5.181909e-04,4.082615,4.453181e-05,‡
5,High_School_Count,0.003499,3.732866e-04,9.374059,6.979578e-21,‡,1.864310e-03,7.439773e-04,2.505870,1.221506e-02,†
6,Log_Dist_Green,0.028799,6.929933e-03,4.155751,3.242208e-05,‡,1.868866e-02,8.279127e-03,2.257322,2.398794e-02,†
7,Log_Dist_Subway,-0.071380,5.943880e-03,-12.009026,3.185732e-33,‡,-3.120162e-02,2.125911e-02,-1.467682,1.421906e-01,
8,Log_Dist_Water,0.004683,5.562308e-03,0.841943,3.998200e-01,,-6.372934e-04,4.697479e-03,-0.135667,8.920845e-01,
9,Median age Population,2.973900,1.506517e-01,19.740231,9.733254e-87,‡,1.774770e+00,6.034153e-01,2.941208,3.269348e-03,‡


,OLS,SLR
--- Property Characteristics ---,,
Size,-0.0001,-0.0005*
Floor,0.0018‡,0.0021‡
Units,0.0001‡,0.0001‡
Parking,0.0185‡,0.0186‡
Construction Year,-0.0020‡,-0.0006
Max Floor,0.0121‡,0.0100‡
Heating Dummy,-0.1783‡,-0.1362‡
--- Accessibility & Environment ---,,
Dist. Subway (Log),-0.0714‡,-0.0312



2022년 결과 저장 완료:
/content/drive/MyDrive/Seoul_Apartment_OLS_SLR_2022.xlsx


2023년 분석 시작
[2023] 결측 제거 전 거래 수: 31071

[2023] 변수별 결측치 상위 10개
Population               0
Sex_ratio                0
Pop. Density             0
Median age Population    0
Young Population         0
Parking_per_Household    0
Log_Dist_Water           0
Log_Dist_Green           0
Log_Dist_Subway          0
Dist_CBD                 0
dtype: int64

[2023] 최종 분석 거래 수: 31071

[2023] 1.0km 공간가중치 생성 중
[경고] 1.0km 이내 다른 좌표의 이웃이 없는 좌표 그룹이 1개이며, 해당 거래는 11건입니다.
[처리] 해당 거래는 제거하지 않고 Wy와 WX를 0으로 처리합니다.
[2023] 유일 좌표 수: 3913
[2023] 좌표 그룹 연결 수: 137748
[2023] 이웃 없는 좌표 그룹 수: 1
[2023] 이웃 없는 거래 수: 11
[2023] 이웃이 있는 좌표의 평균 이웃 좌표 수: 35.21165644171779
[2023] 이웃이 있는 좌표의 평균 이웃 거래 수: 242.43762781186095
[2023] 행 표준화 최대 오차: 8.881784197001252e-16
[2023] 사용된 공간 도구변수 수: 21
[2023] 사용된 공간 도구변수: ['W_Population', 'W_Sex_ratio', 'W_Pop. Density', 'W_Median age Population', 'W_Young Population', 'W_Parking_per_Household', 'W_Log_Dist_Water', 'W_Log_Dis

,Year,Model,N,RMSE,R2,Adjusted_R2,Spatial_Pseudo_R2,AIC,BIC,F_stat,Prob_F,Rho,Rho_pvalue,Robust_SE,Island_Groups,Island_Transactions
0,2023,OLS,31071,0.323257,0.44051,0.440132,NaN,18042.321137,18225.889801,1398.682053,0.0,NaN,NaN,HC0,1,11
1,2023,SLR,31071,0.323715,NaN,NaN,0.438971,NaN,NaN,NaN,NaN,0.222232,0.000001,White,1,11


,Variable,OLS_Coef,OLS_Std_Error,OLS_Statistic,OLS_pvalue,OLS_sig,SLR_Coef,SLR_Std_Error,SLR_Statistic,SLR_pvalue,SLR_sig
0,Bus_Stop,-5.839986e-03,3.386982e-04,-17.242446,1.275253e-66,‡,-4.649414e-03,3.950929e-04,-11.767903,5.712707e-32,‡
1,Construction_Year,-7.033560e-04,2.151970e-04,-3.268429,1.081465e-03,‡,1.408684e-04,2.329245e-04,0.604781,5.453244e-01,
2,Dist_CBD,1.832166e-05,1.317394e-06,13.907506,5.703405e-44,‡,1.304754e-05,1.478124e-06,8.827095,1.074292e-18,‡
3,Fall,-4.282501e-03,5.144026e-03,-0.832519,4.051159e-01,,-2.333439e-03,4.623962e-03,-0.504641,6.138113e-01,
4,Floor,2.753832e-03,3.030169e-04,9.088049,1.008325e-19,‡,2.609380e-03,2.707677e-04,9.636970,5.581113e-22,‡
5,High_School_Count,4.411567e-03,2.092190e-04,21.085883,1.071937e-98,‡,3.254334e-03,2.966523e-04,10.970194,5.315802e-28,‡
6,Log_Dist_Green,3.266799e-02,4.034107e-03,8.097949,5.589356e-16,‡,3.160507e-02,3.703186e-03,8.534560,1.406898e-17,‡
7,Log_Dist_Subway,-1.338727e-01,3.715794e-03,-36.028025,3.046559e-284,‡,-1.018591e-01,7.249445e-03,-14.050606,7.637812e-45,‡
8,Log_Dist_Water,3.880026e-02,3.257307e-03,11.911762,1.027815e-32,‡,2.884469e-02,3.678820e-03,7.840746,4.478764e-15,‡
9,Median age Population,2.859244e+00,9.415880e-02,30.366190,1.536083e-202,‡,2.138264e+00,1.555226e-01,13.748896,5.171233e-43,‡


,OLS,SLR
--- Property Characteristics ---,,
Size,-0.0007‡,-0.0010‡
Floor,0.0028‡,0.0026‡
Units,0.0000‡,0.0000‡
Parking,0.0212‡,0.0204‡
Construction Year,-0.0007‡,0.0001
Max Floor,0.0105‡,0.0097‡
Heating Dummy,-0.1806‡,-0.1507‡
--- Accessibility & Environment ---,,
Dist. Subway (Log),-0.1339‡,-0.1019‡



2023년 결과 저장 완료:
/content/drive/MyDrive/Seoul_Apartment_OLS_SLR_2023.xlsx


전체 분석 완료

2022년 결과 파일:
/content/drive/MyDrive/Seoul_Apartment_OLS_SLR_2022.xlsx

2023년 결과 파일:
/content/drive/MyDrive/Seoul_Apartment_OLS_SLR_2023.xlsx

통합 성능 요약 파일:
/content/drive/MyDrive/Seoul_Apartment_OLS_SLR_2022_2023_Summary.xlsx


In [10]:
# ============================================================
# 추가 셀: 2022년·2023년 OLS / SLR Train-Test 성능 평가
#
# - 연도별로 각각 Train 70% / Test 30% 분할
# - OLS와 SLR에 동일한 분할 사용
# - 모형은 Train 자료로만 추정
# - Train R2 / RMSE와 Test R2 / RMSE 계산
# - SLR Test 예측 시 Test y는 공간시차 계산에 사용하지 않음
# - Test 자료의 좌표와 설명변수만으로 reduced-form 예측
#
# 주의:
# 이 셀을 실행하기 전에 위의 전체 OLS/SLR 코드를 먼저 실행해야 함
# ============================================================

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import r2_score, mean_squared_error
from scipy import sparse
from spreg import TSLS

# ============================================================
# 1. 설정
# ============================================================

EVALUATION_YEARS = [2022, 2023]

TRAIN_RATIO = 0.70
TEST_RATIO = 0.30
RANDOM_STATE = 42

# "random":
# 개별 거래를 무작위로 70:30 분할
#
# "coordinate":
# 동일 좌표 거래가 Train과 Test에 동시에 포함되지 않도록 분할
#
# RF/XGBoost와 일반적인 무작위 분할 조건을 맞추려면 "random" 사용
SPLIT_MODE = "random"

TRAIN_TEST_OUTPUT_DIR = "/content/drive/MyDrive"


# ============================================================
# 2. 기존 코드 실행 여부 확인
# ============================================================

required_objects = [
    "df_all",
    "FEATURE_COLS",
    "TARGET_COL",
    "LAT_COL",
    "LON_COL",
    "DIST_BAND_KM",
    "USE_ROBUST_SE",
    "build_spatial_operator",
    "calculate_spatial_lag",
    "select_valid_instruments",
    "calculate_reduced_form_prediction"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "먼저 위의 전체 OLS/SLR 코드를 실행해야 합니다. "
        f"현재 생성되지 않은 객체: {missing_objects}"
    )


# ============================================================
# 3. RMSE 계산 함수
# ============================================================

def calculate_rmse(actual, predicted):
    actual = np.asarray(actual, dtype=float).reshape(-1)
    predicted = np.asarray(predicted, dtype=float).reshape(-1)

    return np.sqrt(
        mean_squared_error(actual, predicted)
    )


# ============================================================
# 4. 모든 좌표가 이웃이 없을 때 사용할 0 공간가중치
# ============================================================

def build_zero_spatial_operator(
    df_sub,
    lat_col="Latitude",
    lon_col="Longitude"
):
    """
    모든 관측치에 공간이웃이 없는 경우 사용함.

    W = 0으로 설정되므로 다음과 같이 처리됨.

    Wy = 0
    WX = 0
    SLR reduced-form prediction = X beta
    """

    all_coords = df_sub[
        [lat_col, lon_col]
    ].to_numpy(dtype=float)

    unique_coords, coord_id = np.unique(
        all_coords,
        axis=0,
        return_inverse=True
    )

    n_obs = len(df_sub)
    n_groups = len(unique_coords)

    group_counts = np.bincount(
        coord_id,
        minlength=n_groups
    ).astype(float)

    zero_weight_matrix = sparse.csr_matrix(
        (n_groups, n_groups),
        dtype=float
    )

    return {
        "n_obs": n_obs,
        "n_groups": n_groups,
        "coord_id": coord_id,
        "group_counts": group_counts,
        "group_weight_matrix": zero_weight_matrix,

        "neighbor_coordinate_counts": np.zeros(
            n_groups,
            dtype=int
        ),

        "neighbor_transaction_counts": np.zeros(
            n_groups,
            dtype=int
        ),

        "row_sum_max_error": 0.0,
        "n_group_links": 0,

        "island_groups": list(range(n_groups)),
        "n_island_groups": n_groups,
        "n_island_transactions": n_obs
    }


# ============================================================
# 5. 이웃 없는 자료도 허용하는 공간가중치 함수
# ============================================================

def build_spatial_operator_safe(
    df_sub,
    lat_col,
    lon_col,
    dist_band_km
):
    try:
        return build_spatial_operator(
            df_sub=df_sub,
            lat_col=lat_col,
            lon_col=lon_col,
            dist_band_km=dist_band_km
        )

    except ValueError as error:
        error_message = str(error)

        # 모든 좌표가 이웃이 없거나 유일 좌표가 하나뿐인 경우
        if (
            "모든 좌표가" in error_message
            or "유일 좌표가 2개 미만" in error_message
        ):
            print(
                "[경고] 해당 분할 자료에서는 공간이웃을 구성할 수 없어 "
                "모든 관측치의 공간시차를 0으로 처리합니다."
            )

            return build_zero_spatial_operator(
                df_sub=df_sub,
                lat_col=lat_col,
                lon_col=lon_col
            )

        raise


# ============================================================
# 6. Train-Test 분할 함수
# ============================================================

def split_train_test_by_year(
    df_year,
    split_mode="random",
    test_ratio=0.30,
    random_state=42
):
    df_year = df_year.reset_index(drop=True).copy()

    if split_mode == "random":
        train_indices, test_indices = train_test_split(
            np.arange(len(df_year)),
            test_size=test_ratio,
            random_state=random_state,
            shuffle=True
        )

    elif split_mode == "coordinate":
        coordinate_groups = (
            df_year[LAT_COL].astype(str)
            + "_"
            + df_year[LON_COL].astype(str)
        )

        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=test_ratio,
            random_state=random_state
        )

        train_indices, test_indices = next(
            splitter.split(
                df_year,
                groups=coordinate_groups
            )
        )

    else:
        raise ValueError(
            "SPLIT_MODE는 'random' 또는 'coordinate'여야 합니다."
        )

    train_df = (
        df_year
        .iloc[train_indices]
        .copy()
        .reset_index(drop=True)
    )

    test_df = (
        df_year
        .iloc[test_indices]
        .copy()
        .reset_index(drop=True)
    )

    return train_df, test_df


# ============================================================
# 7. 연도별 Train-Test 평가 함수
# ============================================================

def evaluate_train_test_for_year(
    df_input,
    year_value,
    feature_cols,
    target_col,
    lat_col,
    lon_col,
    dist_band_km,
    split_mode="random",
    test_ratio=0.30,
    random_state=42
):
    print("\n")
    print("=" * 80)
    print(f"{year_value}년 Train 70% / Test 30% 평가")
    print("=" * 80)

    required_cols = (
        feature_cols
        + [
            target_col,
            lat_col,
            lon_col
        ]
    )

    df_year = df_input[
        df_input["Analysis_Year"] == year_value
    ].copy()

    df_year["_Original_Index"] = df_year.index

    df_year = (
        df_year
        .dropna(subset=required_cols)
        .reset_index(drop=True)
    )

    if len(df_year) < 30:
        raise ValueError(
            f"{year_value}년 분석자료가 30건 미만입니다."
        )

    # --------------------------------------------------------
    # Train 70% / Test 30% 분할
    # --------------------------------------------------------
    train_df, test_df = split_train_test_by_year(
        df_year=df_year,
        split_mode=split_mode,
        test_ratio=test_ratio,
        random_state=random_state
    )

    print(f"[{year_value}] 전체 자료 수:", len(df_year))
    print(f"[{year_value}] Train 자료 수:", len(train_df))
    print(f"[{year_value}] Test 자료 수:", len(test_df))
    print(f"[{year_value}] 분할 방식:", split_mode)

    # --------------------------------------------------------
    # 설명변수와 종속변수
    # --------------------------------------------------------
    x_train = train_df[
        feature_cols
    ].to_numpy(dtype=float)

    x_test = test_df[
        feature_cols
    ].to_numpy(dtype=float)

    y_train = train_df[
        target_col
    ].to_numpy(dtype=float).reshape(-1)

    y_test = test_df[
        target_col
    ].to_numpy(dtype=float).reshape(-1)

    # ========================================================
    # 8. OLS Train 추정
    # ========================================================

    x_train_ols = sm.add_constant(
        train_df[feature_cols],
        has_constant="add"
    )

    x_test_ols = sm.add_constant(
        test_df[feature_cols],
        has_constant="add"
    )

    # Train과 Test의 열 순서 일치
    x_test_ols = x_test_ols.reindex(
        columns=x_train_ols.columns
    )

    if USE_ROBUST_SE:
        ols_model = sm.OLS(
            y_train,
            x_train_ols
        ).fit(
            cov_type="HC0"
        )

    else:
        ols_model = sm.OLS(
            y_train,
            x_train_ols
        ).fit()

    ols_train_prediction = np.asarray(
        ols_model.predict(x_train_ols)
    ).reshape(-1)

    ols_test_prediction = np.asarray(
        ols_model.predict(x_test_ols)
    ).reshape(-1)

    ols_train_r2 = r2_score(
        y_train,
        ols_train_prediction
    )

    ols_test_r2 = r2_score(
        y_test,
        ols_test_prediction
    )

    ols_train_rmse = calculate_rmse(
        y_train,
        ols_train_prediction
    )

    ols_test_rmse = calculate_rmse(
        y_test,
        ols_test_prediction
    )

    # ========================================================
    # 9. SLR Train 공간가중치 생성
    # ========================================================

    print(
        f"\n[{year_value}] Train 공간가중치 생성 중"
    )

    train_spatial_info = build_spatial_operator_safe(
        df_sub=train_df,
        lat_col=lat_col,
        lon_col=lon_col,
        dist_band_km=dist_band_km
    )

    if train_spatial_info["n_group_links"] == 0:
        raise ValueError(
            f"{year_value}년 Train 자료에 공간이웃이 전혀 없어 "
            "SLR을 추정할 수 없습니다."
        )

    # Train Wy
    wy_train = calculate_spatial_lag(
        y_train,
        train_spatial_info
    ).reshape(-1, 1)

    # Train WX
    wx_train = calculate_spatial_lag(
        x_train,
        train_spatial_info
    )

    wx_names = [
        f"W_{col}"
        for col in feature_cols
    ]

    q_train, q_names = select_valid_instruments(
        x_array=x_train,
        q_array=wx_train,
        q_names=wx_names
    )

    print(
        f"[{year_value}] Train 공간 도구변수 수:",
        q_train.shape[1]
    )

    # ========================================================
    # 10. SLR Train 모형 추정
    # ========================================================

    slr_model = TSLS(
        y=y_train.reshape(-1, 1),
        x=x_train,
        yend=wy_train,
        q=q_train,

        robust=(
            "white"
            if USE_ROBUST_SE
            else None
        ),

        name_y=target_col,
        name_x=feature_cols,
        name_yend=[f"W_{target_col}"],
        name_q=q_names,
        name_ds=f"Seoul Apartment {year_value} Train"
    )

    slr_betas = np.asarray(
        slr_model.betas
    ).reshape(-1)

    slr_intercept = slr_betas[0]

    slr_feature_betas = slr_betas[
        1:1 + len(feature_cols)
    ]

    estimated_rho = float(
        slr_betas[-1]
    )

    print(
        f"[{year_value}] Train에서 추정된 rho:",
        round(estimated_rho, 6)
    )

    if abs(estimated_rho) >= 1:
        raise ValueError(
            f"{year_value}년 추정 rho={estimated_rho:.6f}가 "
            "|rho|<1 범위를 벗어났습니다."
        )

    # ========================================================
    # 11. SLR Train 예측
    # ========================================================

    xb_train = (
        slr_intercept
        + x_train @ slr_feature_betas
    )

    slr_train_prediction = calculate_reduced_form_prediction(
        xb=xb_train,
        rho=estimated_rho,
        spatial_info=train_spatial_info
    )

    slr_train_r2 = r2_score(
        y_train,
        slr_train_prediction
    )

    slr_train_rmse = calculate_rmse(
        y_train,
        slr_train_prediction
    )

    # ========================================================
    # 12. SLR Test 예측
    #
    # Test y는 공간시차에 사용하지 않음.
    # Test 좌표와 Test X로 W_test를 구성하고,
    # Train에서 추정한 beta와 rho를 적용함.
    # ========================================================

    print(
        f"[{year_value}] Test 공간가중치 생성 중"
    )

    test_spatial_info = build_spatial_operator_safe(
        df_sub=test_df,
        lat_col=lat_col,
        lon_col=lon_col,
        dist_band_km=dist_band_km
    )

    xb_test = (
        slr_intercept
        + x_test @ slr_feature_betas
    )

    slr_test_prediction = calculate_reduced_form_prediction(
        xb=xb_test,
        rho=estimated_rho,
        spatial_info=test_spatial_info
    )

    slr_test_r2 = r2_score(
        y_test,
        slr_test_prediction
    )

    slr_test_rmse = calculate_rmse(
        y_test,
        slr_test_prediction
    )

    # ========================================================
    # 13. 성능표
    # ========================================================

    performance_table = pd.DataFrame([
        {
            "Year": year_value,
            "Model": "OLS",
            "Split": "Train",
            "N": len(train_df),
            "R2": ols_train_r2,
            "RMSE": ols_train_rmse,
            "Estimated_Rho": np.nan,
            "Split_Mode": split_mode,
            "Random_State": random_state
        },
        {
            "Year": year_value,
            "Model": "OLS",
            "Split": "Test",
            "N": len(test_df),
            "R2": ols_test_r2,
            "RMSE": ols_test_rmse,
            "Estimated_Rho": np.nan,
            "Split_Mode": split_mode,
            "Random_State": random_state
        },
        {
            "Year": year_value,
            "Model": "SLR",
            "Split": "Train",
            "N": len(train_df),
            "R2": slr_train_r2,
            "RMSE": slr_train_rmse,
            "Estimated_Rho": estimated_rho,
            "Split_Mode": split_mode,
            "Random_State": random_state
        },
        {
            "Year": year_value,
            "Model": "SLR",
            "Split": "Test",
            "N": len(test_df),
            "R2": slr_test_r2,
            "RMSE": slr_test_rmse,
            "Estimated_Rho": estimated_rho,
            "Split_Mode": split_mode,
            "Random_State": random_state
        }
    ])

    # ========================================================
    # 14. 예측값 저장표
    # ========================================================

    train_prediction_table = pd.DataFrame({
        "Year": year_value,
        "Split": "Train",

        "Original_Index": train_df[
            "_Original_Index"
        ].values,

        "Actual": y_train,

        "OLS_Prediction": ols_train_prediction,
        "OLS_Residual": (
            y_train - ols_train_prediction
        ),

        "SLR_Prediction": slr_train_prediction,
        "SLR_Residual": (
            y_train - slr_train_prediction
        ),

        "Latitude": train_df[
            lat_col
        ].values,

        "Longitude": train_df[
            lon_col
        ].values
    })

    test_prediction_table = pd.DataFrame({
        "Year": year_value,
        "Split": "Test",

        "Original_Index": test_df[
            "_Original_Index"
        ].values,

        "Actual": y_test,

        "OLS_Prediction": ols_test_prediction,
        "OLS_Residual": (
            y_test - ols_test_prediction
        ),

        "SLR_Prediction": slr_test_prediction,
        "SLR_Residual": (
            y_test - slr_test_prediction
        ),

        "Latitude": test_df[
            lat_col
        ].values,

        "Longitude": test_df[
            lon_col
        ].values
    })

    prediction_table = pd.concat(
        [
            train_prediction_table,
            test_prediction_table
        ],
        ignore_index=True
    )

    print(f"\n[{year_value}] Train-Test 성능")
    display(performance_table)

    return {
        "year": year_value,
        "performance_table": performance_table,
        "prediction_table": prediction_table,
        "ols_model": ols_model,
        "slr_model": slr_model,
        "train_n": len(train_df),
        "test_n": len(test_df),
        "rho": estimated_rho,
        "instrument_names": q_names
    }


# ============================================================
# 15. 2022년과 2023년 각각 실행
# ============================================================

all_train_test_performance = []

for evaluation_year in EVALUATION_YEARS:

    result = evaluate_train_test_for_year(
        df_input=df_all,
        year_value=evaluation_year,

        feature_cols=FEATURE_COLS,

        target_col=TARGET_COL,
        lat_col=LAT_COL,
        lon_col=LON_COL,

        dist_band_km=DIST_BAND_KM,

        split_mode=SPLIT_MODE,
        test_ratio=TEST_RATIO,
        random_state=RANDOM_STATE
    )

    year_output_file = os.path.join(
        TRAIN_TEST_OUTPUT_DIR,
        f"Seoul_Apartment_OLS_SLR_Train_Test_{evaluation_year}.xlsx"
    )

    with pd.ExcelWriter(
        year_output_file,
        engine="openpyxl"
    ) as writer:

        result[
            "performance_table"
        ].to_excel(
            writer,
            sheet_name="Train_Test_Performance",
            index=False
        )

        result[
            "prediction_table"
        ].to_excel(
            writer,
            sheet_name="Predictions",
            index=False
        )

        pd.DataFrame({
            "Selected_Instrument": result[
                "instrument_names"
            ]
        }).to_excel(
            writer,
            sheet_name="Instruments",
            index=False
        )

        pd.DataFrame({
            "OLS_Summary": str(
                result["ols_model"].summary()
            ).splitlines()
        }).to_excel(
            writer,
            sheet_name="OLS_Train_Summary",
            index=False
        )

        pd.DataFrame({
            "SLR_Summary": str(
                result["slr_model"].summary
            ).splitlines()
        }).to_excel(
            writer,
            sheet_name="SLR_Train_Summary",
            index=False
        )

    all_train_test_performance.append(
        result["performance_table"].copy()
    )

    print(
        f"\n{evaluation_year}년 Train-Test 결과 저장 완료:"
    )
    print(year_output_file)


# ============================================================
# 16. 2022년·2023년 통합 성능표
# ============================================================

combined_train_test_performance = pd.concat(
    all_train_test_performance,
    ignore_index=True
)

combined_train_test_output = os.path.join(
    TRAIN_TEST_OUTPUT_DIR,
    "Seoul_Apartment_OLS_SLR_Train_Test_2022_2023.xlsx"
)

with pd.ExcelWriter(
    combined_train_test_output,
    engine="openpyxl"
) as writer:

    combined_train_test_performance.to_excel(
        writer,
        sheet_name="Combined_Performance",
        index=False
    )


# ============================================================
# 17. 최종 결과 출력
# ============================================================

print("\n")
print("=" * 80)
print("2022년·2023년 Train-Test 평가 완료")
print("=" * 80)

display(combined_train_test_performance)

print("\n2022년 결과:")
print(
    os.path.join(
        TRAIN_TEST_OUTPUT_DIR,
        "Seoul_Apartment_OLS_SLR_Train_Test_2022.xlsx"
    )
)

print("\n2023년 결과:")
print(
    os.path.join(
        TRAIN_TEST_OUTPUT_DIR,
        "Seoul_Apartment_OLS_SLR_Train_Test_2023.xlsx"
    )
)

print("\n통합 결과:")
print(combined_train_test_output)

print("=" * 80)



2022년 Train 70% / Test 30% 평가
[2022] 전체 자료 수: 10579
[2022] Train 자료 수: 7405
[2022] Test 자료 수: 3174
[2022] 분할 방식: random

[2022] Train 공간가중치 생성 중
[경고] 1.0km 이내 다른 좌표의 이웃이 없는 좌표 그룹이 2개이며, 해당 거래는 2건입니다.
[처리] 해당 거래는 제거하지 않고 Wy와 WX를 0으로 처리합니다.
[2022] Train 공간 도구변수 수: 21
TSLS
[2022] Train에서 추정된 rho: 0.219018
[2022] Test 공간가중치 생성 중
[경고] 1.0km 이내 다른 좌표의 이웃이 없는 좌표 그룹이 3개이며, 해당 거래는 3건입니다.
[처리] 해당 거래는 제거하지 않고 Wy와 WX를 0으로 처리합니다.

[2022] Train-Test 성능


,Year,Model,Split,N,R2,RMSE,Estimated_Rho,Split_Mode,Random_State
0,2022,OLS,Train,7405,0.455821,0.331810,NaN,random,42
1,2022,OLS,Test,3174,0.438735,0.342981,NaN,random,42
2,2022,SLR,Train,7405,0.457580,0.331273,0.219018,random,42
3,2022,SLR,Test,3174,0.409014,0.351945,0.219018,random,42



2022년 Train-Test 결과 저장 완료:
/content/drive/MyDrive/Seoul_Apartment_OLS_SLR_Train_Test_2022.xlsx


2023년 Train 70% / Test 30% 평가
[2023] 전체 자료 수: 31071
[2023] Train 자료 수: 21749
[2023] Test 자료 수: 9322
[2023] 분할 방식: random

[2023] Train 공간가중치 생성 중
[경고] 1.0km 이내 다른 좌표의 이웃이 없는 좌표 그룹이 2개이며, 해당 거래는 11건입니다.
[처리] 해당 거래는 제거하지 않고 Wy와 WX를 0으로 처리합니다.
[2023] Train 공간 도구변수 수: 21
TSLS
[2023] Train에서 추정된 rho: 0.160481
[2023] Test 공간가중치 생성 중
[경고] 1.0km 이내 다른 좌표의 이웃이 없는 좌표 그룹이 4개이며, 해당 거래는 35건입니다.
[처리] 해당 거래는 제거하지 않고 Wy와 WX를 0으로 처리합니다.

[2023] Train-Test 성능


,Year,Model,Split,N,R2,RMSE,Estimated_Rho,Split_Mode,Random_State
0,2023,OLS,Train,21749,0.442834,0.322184,NaN,random,42
1,2023,OLS,Test,9322,0.434461,0.325942,NaN,random,42
2,2023,SLR,Train,21749,0.441484,0.322574,0.160481,random,42
3,2023,SLR,Test,9322,0.316782,0.358252,0.160481,random,42



2023년 Train-Test 결과 저장 완료:
/content/drive/MyDrive/Seoul_Apartment_OLS_SLR_Train_Test_2023.xlsx


2022년·2023년 Train-Test 평가 완료


,Year,Model,Split,N,R2,RMSE,Estimated_Rho,Split_Mode,Random_State
0,2022,OLS,Train,7405,0.455821,0.331810,NaN,random,42
1,2022,OLS,Test,3174,0.438735,0.342981,NaN,random,42
2,2022,SLR,Train,7405,0.457580,0.331273,0.219018,random,42
3,2022,SLR,Test,3174,0.409014,0.351945,0.219018,random,42
4,2023,OLS,Train,21749,0.442834,0.322184,NaN,random,42
5,2023,OLS,Test,9322,0.434461,0.325942,NaN,random,42
6,2023,SLR,Train,21749,0.441484,0.322574,0.160481,random,42
7,2023,SLR,Test,9322,0.316782,0.358252,0.160481,random,42



2022년 결과:
/content/drive/MyDrive/Seoul_Apartment_OLS_SLR_Train_Test_2022.xlsx

2023년 결과:
/content/drive/MyDrive/Seoul_Apartment_OLS_SLR_Train_Test_2023.xlsx

통합 결과:
/content/drive/MyDrive/Seoul_Apartment_OLS_SLR_Train_Test_2022_2023.xlsx


In [11]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

year_check = 2022

required_cols = FEATURE_COLS + [
    TARGET_COL,
    LAT_COL,
    LON_COL
]

check_df = (
    df_all[df_all["Analysis_Year"] == year_check]
    .dropna(subset=required_cols)
    .copy()
)

vif_x = sm.add_constant(
    check_df[FEATURE_COLS],
    has_constant="add"
)

vif_table = pd.DataFrame({
    "Variable": vif_x.columns,
    "VIF": [
        variance_inflation_factor(
            vif_x.to_numpy(dtype=float),
            i
        )
        for i in range(vif_x.shape[1])
    ]
})

display(
    vif_table.sort_values(
        "VIF",
        ascending=False
    )
)

,Variable,VIF
0,const,48362.396441
11,max_floor,2.058908
5,Young Population,1.892499
19,Spring,1.782862
21,Winter,1.637497
13,num_of_people,1.568867
4,Median age Population,1.499893
12,heating_dummy,1.498631
20,Fall,1.489316
17,Size_m2,1.425231
